# Pre-norm vs Post-norm

This note explains what **pre-norm** and **post-norm** mean in a Transformer block, why the distinction matters, and when each design is preferred in practice.

---

## 1. What changes between the two designs?

A Transformer block contains three important pieces:

- a **residual connection**
- a **sublayer** such as self-attention or the MLP
- a **LayerNorm**

The only difference between pre-norm and post-norm is **where LayerNorm is applied relative to the residual addition**.

### Pre-norm

LayerNorm is applied **before** the sublayer:

$$
x_{l+1} = x_l + F(\mathrm{LN}(x_l))
$$

Read this as:

1. normalize the current representation
2. pass it through the sublayer
3. add the result back to the original residual stream

### Post-norm

LayerNorm is applied **after** the residual addition:

$$
x_{l+1} = \mathrm{LN}(x_l + F(x_l))
$$

Read this as:

1. pass the current representation through the sublayer
2. add it back to the residual stream
3. normalize the combined result

---

## 2. Simple intuition

A good mental model is to think of the **residual stream** as a highway carrying information through the network.

- In **pre-norm**, each layer computes an update and adds it onto that highway, while the highway itself stays relatively clean.
- In **post-norm**, after adding the update, the whole highway is normalized again.

That difference matters because deep Transformers rely heavily on the residual path for stable information flow and stable gradients.

---

## 3. Why pre-norm is usually easier to optimize

The main reason is the **gradient path**.

For **pre-norm**,

$$
x_{l+1} = x_l + F(\mathrm{LN}(x_l))
$$

Differentiate with respect to $x_l$:

$$
\frac{\partial x_{l+1}}{\partial x_l} = I + \frac{\partial F(\mathrm{LN}(x_l))}{\partial x_l}
$$

So the Jacobian looks like:

$$
I + J
$$

where $J$ is the Jacobian of the normalized sublayer.

This is the key idea:

- the residual branch contributes a clean **identity term** $I$
- the sublayer contributes a **perturbation** $J$
- so gradients can always use the identity path, even if the sublayer is badly conditioned early in training

This does **not** mean gradients are guaranteed to be perfect. If $J$ becomes very large, training can still be unstable. But empirically, having the explicit identity path makes optimization much more robust.

### Why this helps in deep models

When a network has many layers, gradients are repeatedly multiplied by layer Jacobians.

- If each layer is roughly like $I + J$, the gradient can keep flowing through the identity component.
- That makes vanishing and exploding gradients less severe in practice.
- This is especially valuable when scaling Transformers to many layers.

---

## 4. Why post-norm is harder

For **post-norm**,

$$
x_{l+1} = \mathrm{LN}(x_l + F(x_l))
$$

Differentiate with respect to $x_l$:

$$
\frac{\partial x_{l+1}}{\partial x_l}
=
\frac{\partial \mathrm{LN}(x_l + F(x_l))}{\partial (x_l + F(x_l))}
\left(I + \frac{\partial F(x_l)}{\partial x_l}\right)
$$

Now the shortcut is no longer a clean identity path, because it is multiplied by the **LayerNorm Jacobian** after the residual merge.

That means:

- the residual signal is altered after every layer
- the direct path for gradients is less clean
- optimization becomes more fragile as depth increases

So the core issue is not that post-norm is mathematically wrong. The issue is that it is **harder to train reliably**, especially in deep networks.

---

## 5. Residual-path interpretation

Another way to think about it:

### Pre-norm

Each layer says:

> Take a normalized view of the current state, compute an update, and add that update back to the running representation.

This tends to preserve the residual stream more cleanly.

### Post-norm

Each layer says:

> Compute an update, merge it with the residual stream, and then normalize the whole result.

This means the residual stream is re-shaped after every block.

That repeated reshaping is part of why post-norm becomes harder to optimize at large depth.

---

## 6. Pros and cons

| Factor | Pre-norm | Post-norm |
| --- | --- | --- |
| Training stability | Usually more stable | More sensitive |
| Gradient flow | Cleaner residual path | Residual path is modified by LN |
| Depth scaling | Better for deep stacks | Harder as depth grows |
| Hyperparameter tuning | Usually easier | Usually needs more care |
| Learning-rate sensitivity | Often more forgiving | More likely to need warmup/tuning |
| Final performance | Reliable and consistent | Can sometimes be slightly better if tuned well |
| Practical use in modern LLMs | Very common | Less common in large deep models |

### Pre-norm: advantages

- Better gradient flow in deep networks.
- More stable optimization.
- Easier training with less hyperparameter tuning.
- Better default choice for large or deep Transformer models.
- Often preferred when engineering reliability matters.

### Pre-norm: limitations

- It does not guarantee perfect stability.
- A badly behaved sublayer can still produce large Jacobian terms.
- In some carefully tuned settings, final quality may be slightly below a strong post-norm run.

### Post-norm: advantages

- Used in the original Transformer paper.
- Can work well when training is carefully controlled.
- Sometimes achieves slightly better final performance.
- Useful when the model is not extremely deep and careful tuning is acceptable.

### Post-norm: limitations

- Harder to optimize, especially for deep models.
- More sensitive to learning rate, warmup, and initialization.
- Requires more care to avoid unstable training.
- Less attractive when compute budget or iteration speed is limited.

---

## 7. Historical note

The original **Attention Is All You Need** Transformer used **post-norm**.

As Transformers became deeper and larger, many later architectures shifted toward **pre-norm** because it made optimization much easier and more reliable.

A useful historical summary is:

- **original Transformer**: post-norm
- **many modern large Transformer families**: pre-norm or related stable residual designs

The shift happened mainly because practitioners cared more about stable deep optimization than about preserving the exact original design.

---

## 8. When should you choose pre-norm?

Choose **pre-norm** when:

- the model is deep
- you want stable training with fewer tricks
- you have limited compute budget for repeated tuning
- you care about reliable convergence
- you are building or studying modern large language models

In practice, **pre-norm is usually the safer default**.

---

## 9. When should you choose post-norm?

Choose **post-norm** when:

- the model is not extremely deep
- the training pipeline is well controlled
- you can afford careful hyperparameter tuning
- you want to explore whether a better final optimum is possible
- maximum final performance matters more than ease of optimization

So post-norm is not obsolete. It is just **less forgiving**.

---

## 10. Common misconception

A common misunderstanding is:

> Pre-norm is stable because the gradient is exactly $1 + \epsilon$.

That is only a simplified scalar intuition.

The more correct statement is:

- scalar intuition: $1 + \epsilon$
- vector case: $I + J$

So pre-norm does **not** make the gradient exactly stable in a strict mathematical sense. It simply gives the network a cleaner identity route, which makes stable training much more likely in practice.

---

## 11. Interview-safe takeaway

A strong interview answer is:

> Pre-norm is usually easier to train because it preserves a cleaner identity residual path. The block Jacobian is closer to $I +$ a perturbation, so gradients flow more directly across many layers. Post-norm puts LayerNorm after the residual merge, which distorts that shortcut and makes deep optimization harder.

---

## 12. Final summary

- **Pre-norm** places LayerNorm before the sublayer and usually gives better gradient flow.
- **Post-norm** places LayerNorm after the residual addition and can be harder to optimize.
- **Pre-norm** is usually preferred for deep modern Transformers because it is more stable.
- **Post-norm** can still be useful when training is well controlled and final performance is the main priority.

If you only remember one sentence, remember this:

**Pre-norm is usually the modern practical default because it keeps the residual path cleaner, which makes deep Transformer training more stable.**

## 13. Visual block structure

Sometimes the easiest way to remember the difference is to look at the order of operations.

### Pre-norm block

```text
x
| 
|----> residual ---------------------------+
|                                          |
+--> LayerNorm --> Sublayer F(x) --------->+
                                           |
                                           v
                                        output
```

Formula:

$$
x_{l+1} = x_l + F(\mathrm{LN}(x_l))
$$

Interpretation:

- normalize first
- compute an update
- add the update to the original residual stream

### Post-norm block

```text
x
| 
|----> residual -------------------+
|                                  |
+--> Sublayer F(x) ----------------+
                                   |
                                   v
                               LayerNorm
                                   |
                                   v
                                output
```

Formula:

$$
x_{l+1} = \mathrm{LN}(x_l + F(x_l))
$$

Interpretation:

- compute an update first
- merge with the residual stream
- normalize the merged result

### Visual takeaway

In **pre-norm**, the residual path stays cleaner.
In **post-norm**, the residual stream is renormalized after the merge.
That is the structural reason the two designs behave differently during optimization.

In [ ]:
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            nn.GELU(),
            nn.Linear(d_hidden, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class PreNormBlock(nn.Module):
    def __init__(self, d_model: int, d_hidden: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.ffn(self.norm(x))


class PostNormBlock(nn.Module):
    def __init__(self, d_model: int, d_hidden: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(x + self.ffn(x))


x = torch.randn(2, 4, 8)
pre_block = PreNormBlock(d_model=8, d_hidden=16)
post_block = PostNormBlock(d_model=8, d_hidden=16)

pre_out = pre_block(x)
post_out = post_block(x)

print('input shape:', x.shape)
print('pre-norm output shape:', pre_out.shape)
print('post-norm output shape:', post_out.shape)

## 14. Check your understanding

Use these questions to verify that the core idea is clear.

### Quick questions

1. In pre-norm, where is LayerNorm applied?
2. Why is the Jacobian of a pre-norm block often described as being close to $I + J$?
3. Why does post-norm usually become harder to optimize as depth increases?
4. Which design is usually the safer default for modern deep Transformers?
5. When might post-norm still be worth trying?

### Short answers

1. **Before** the sublayer, inside the residual block.
2. Because the residual branch contributes an explicit identity term, while the sublayer contributes a perturbation.
3. Because the residual shortcut is no longer clean; it is modified by LayerNorm after the residual merge.
4. **Pre-norm**, because it usually gives more stable optimization.
5. When the model is not extremely deep and you can afford careful tuning for possibly better final performance.

### One-sentence memory trick

- **Pre-norm**: normalize first, then add an update to a cleaner residual stream.
- **Post-norm**: add first, then normalize the merged result.

### Interview version

If asked in an interview, a concise answer is:

> Pre-norm is usually easier to train because it keeps a cleaner identity residual path, so gradients flow more directly through deep layers. Post-norm applies LayerNorm after the residual merge, which distorts that shortcut and makes optimization less stable as depth increases.